In [0]:
fetch_date = dbutils.widgets.get("fetch_date")
date_table = dbutils.widgets.get("date_table")
target_table = dbutils.widgets.get("target_table")

In [0]:
max_date = spark.sql(f"""
SELECT MAX(CalendarDate) AS MaxDate
FROM {date_table}
WHERE FiscalQuarterNbr = (
    SELECT FiscalQuarterNbr
    FROM {date_table}
    WHERE CalendarDate = CAST('{fetch_date}' AS DATE)
)
GROUP BY FiscalQuarterNbr
""").collect()[0]['MaxDate']

In [0]:
display(
spark.sql(f""" 
CREATE OR REPLACE TEMP VIEW bad_debt_src AS
SELECT * FROM (      
-- CTE 1: Build temp table with Pre_Bad_Debt_Date calculation
WITH tmpma AS (
    SELECT 
        Client_Number,
        Invoice_Number,
        Account_Balance,
        Payer_Type,
        Bill_Date,
        Claim_From_Date,
        (COALESCE(Total_Positive_Client_Balance, 0) + COALESCE(Total_Negative_Client_Balance, 0)) AS Net_Client_Credit_Balance,
        CASE 
            WHEN Reimbursement_Team IN ('413', '415', '596', '1375', '1376') 
            THEN DATE_ADD(Bill_Date, 270)
            WHEN Reimbursement_Team = '289' 
                AND Payer_Type IN ('HOSPICE - MEDICARE', 'MEDICARE', 'MEDICARE - PART B')
            THEN DATE_ADD(Claim_From_Date, 180)
            WHEN Reimbursement_Team = '289' 
                AND Payer_Type NOT IN ('HOSPICE - MEDICARE', 'MEDICARE', 'MEDICARE - PART B')
            THEN DATE_ADD(Claim_From_Date, 270)
        END AS Pre_Bad_Debt_Date
    FROM {target_table}
    WHERE reporting_date =  CAST('{fetch_date}' AS DATE)
),

-- CTE 2: Get max week ending date per fiscal quarter
max_week_ending AS (
    SELECT 
        FiscalQuarterNbr,
        MAX(WeekEndingDate) AS MaxWeekEndingDateInQuarter
    FROM {date_table}
    GROUP BY FiscalQuarterNbr
),

-- CTE 3: Join with date dimension to get fiscal quarter and bad debt date
tmpma_with_date AS (
    SELECT 
        m.*,
        d.FiscalQuarterNbr,
        c.MaxWeekEndingDateInQuarter AS Bad_Debt_Date
    FROM tmpma m
    INNER JOIN {date_table} d
        ON m.Pre_Bad_Debt_Date = d.CalendarDate
    LEFT JOIN max_week_ending c
        ON c.FiscalQuarterNbr = d.FiscalQuarterNbr
),

-- CTE 4: Calculate Bad Debt Amount
tmpma_with_amount AS (
    SELECT 
        *,
        CASE 
            WHEN Bad_Debt_Date <= DATE('{max_date}')  
            AND Net_Client_Credit_Balance > 0 
            THEN Account_Balance
        END AS Bad_Debt_Amount
    FROM tmpma_with_date
),

-- CTE 5: Calculate Bad Debt Indicator
tmpma_final AS (
    SELECT 
        *,
        CASE 
            WHEN Bad_Debt_Date > DATE('{max_date}')
            THEN 'X'
            WHEN Bad_Debt_Date <= DATE('{max_date}')  
                AND Bad_Debt_Amount != 0 
            THEN 'Y'
            ELSE 'Z'
        END AS Bad_Debt_Indicator
    FROM tmpma_with_amount
)
SELECT * FROM tmpma_final
)
""")
)

In [0]:
display(
spark.sql(f"""  
-- Step 1: Update Master Aging with Bad Debt calculations
MERGE INTO {target_table} AS target
USING bad_debt_src AS source
ON target.Invoice_Number = source.Invoice_Number
   AND target.reporting_date = CAST('{fetch_date}' AS DATE)

WHEN MATCHED THEN
    UPDATE SET
        target.Bad_Debt_Date = source.Bad_Debt_Date,
        target.Bad_Debt_Amount_Preliminary = source.Bad_Debt_Amount,
        target.Bad_Debt_Indicator_Preliminary = source.Bad_Debt_Indicator;
""")
)

In [0]:
display(
spark.sql(f""" 
-- Step 2: Set default Bad Debt Date for NULL values
UPDATE {target_table}
SET Bad_Debt_Date = '2007-01-01'
WHERE Bad_Debt_Date IS NULL
AND reporting_date = CAST('{fetch_date}' AS DATE);
""")
)

In [0]:
%skip
display(
spark.sql(f""" 
-- Step 3: Clear NULL Bad Debt Amount values
UPDATE dev_bronze_landing.alphacollector.master_aging
SET Bad_Debt_Amount_Preliminary = ' '
WHERE Bad_Debt_Amount_Preliminary IS NULL
AND reporting_date = CAST('{fetch_date}' AS DATE);
""")
)